# Espresso puck permeability — lattice-Boltzmann sweeps (Colab)

Self-contained: run top to bottom. **Runtime → Change runtime type → T4 GPU** first.

Contents: validated D3Q19 TRT solver (`taichi_lb.py`), synthetic-pack generator +
streamtube-column σ pipeline (`puck_pack.py`), a validation gate, and two sweeps:

* **Sweep A** — synthetic replication of Wadsworth et al. 2026 (RSOS 252031) Table 1:
  overlapping-sphere packs at their measured connected porosity and mean grain radius,
  solved for permeability, compared against their XCT/LBflow measurements.
* **Sweep B** — σ_micro production study: heterogeneity amplitude × grind setting →
  streamtube-column flux spread, against the paper's calibrated σ* = 0.57 / 0.36 / 0.22.

Precision note: `f32` + driving force `g = 1e-4` is the validated Colab configuration
(0.03% vs the f64 reference; Stokes linearity makes the stronger force free).
Set `QUICK = False` for the full grids once the quick pass looks right.

In [ ]:
%%capture
!pip install taichi tifffile

In [ ]:
%%writefile taichi_lb.py
"""taichi_lb.py — Taichi port of the validated mini_lb D3Q19 TRT permeability kernel.

Same physics as mini_lb.py (validated: plane Poiseuille +0.03%; SC sphere
arrays -7..-8% at toy radii, converging ~1/a). One-line switch between CPU
(analysis/testing, e.g. in a Claude session) and GPU (production on a local
card): change ARCH below or pass arch= to init_lb().

    import taichi_lb as tlb
    tlb.init_lb(arch="gpu")                    # or "cpu"
    r = tlb.solve(solid, g=1e-6, tau_plus=1.5) # solid: numpy bool (L,L,L)
    r["k"], r["ux"]                            # permeability (lu), velocity field

API and conventions match mini_lb.solve() so the two kernels cross-check:
periodic box, full-way bounce-back, constant body force in +x, TRT with
magic Lambda = 3/16, raw-velocity equilibrium + source-term forcing,
half-force correction applied at measurement only.
"""
import numpy as np
import time

import taichi as ti

# ---------------- D3Q19 lattice (identical ordering to mini_lb) ----------------
C_np = np.array([
    [0,0,0],
    [1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1],
    [1,1,0],[-1,-1,0],[1,-1,0],[-1,1,0],
    [1,0,1],[-1,0,-1],[1,0,-1],[-1,0,1],
    [0,1,1],[0,-1,-1],[0,1,-1],[0,-1,1]], dtype=np.int32)
W_np = np.array([1/3] + [1/18]*6 + [1/36]*12)
OPP_np = np.array([0,2,1,4,3,6,5,8,7,10,9,12,11,14,13,16,15,18,17], dtype=np.int32)
Q = 19

_state = {}

def init_lb(arch="cpu", dtype="f64"):
    """arch: 'cpu' or 'gpu' (CUDA/Vulkan/Metal via ti.gpu). dtype: 'f64' or 'f32'.
    f32 halves memory (Colab T4: ~300^3 domains) but requires a stronger driving
    force to stay above rounding: use g ~ 1e-4 (Stokes linearity makes this free;
    Ma ~ 1e-3, Re << 1). Re-initializes the Taichi runtime, freeing prior fields —
    sweep drivers should call this once per run to avoid field accumulation."""
    fp = ti.f64 if dtype == "f64" else ti.f32
    ti.init(arch=ti.gpu if arch == "gpu" else ti.cpu, default_fp=fp)
    _state["dtype"] = fp
    _state["ready"] = True

def solve(solid_np, g=1e-6, tau_plus=1.5, max_steps=20000, check=100,
          rtol=1e-6, min_steps=1500, verbose=True):
    if not _state.get("ready"):
        init_lb()
    L = solid_np.shape[0]
    assert solid_np.shape == (L, L, L), "cubic domains only in this port"
    fdt = _state["dtype"]

    nu = (tau_plus - 0.5) / 3.0
    lam = 3.0/16.0
    tau_minus = 0.5 + lam/(tau_plus - 0.5)
    om_p, om_m = 1.0/tau_plus, 1.0/tau_minus

    f     = ti.field(fdt, shape=(Q, L, L, L))
    fpost = ti.field(fdt, shape=(Q, L, L, L))
    solid = ti.field(ti.i8, shape=(L, L, L))
    ux    = ti.field(fdt, shape=(L, L, L))
    C   = ti.Vector.field(3, ti.i32, shape=Q)
    Wt  = ti.field(fdt, shape=Q)
    OPP = ti.field(ti.i32, shape=Q)
    C.from_numpy(C_np); Wt.from_numpy(W_np); OPP.from_numpy(OPP_np)
    solid.from_numpy(solid_np.astype(np.int8))

    @ti.kernel
    def init_eq():
        for x, y, z in ti.ndrange(L, L, L):
            for i in ti.static(range(Q)):
                f[i, x, y, z] = Wt[i]

    @ti.kernel
    def collide():
        for x, y, z in ti.ndrange(L, L, L):
            if solid[x, y, z] == 1:
                for i in ti.static(range(Q)):
                    fpost[i, x, y, z] = f[i, x, y, z]
            else:
                rho = 0.0
                u = ti.Vector([0.0, 0.0, 0.0])
                for i in ti.static(range(Q)):
                    fi = f[i, x, y, z]
                    rho += fi
                    u += fi * ti.Vector([ti.cast(C[i][0], fdt),
                                         ti.cast(C[i][1], fdt),
                                         ti.cast(C[i][2], fdt)])
                u /= rho
                u2 = u.dot(u)
                for i in ti.static(range(Q)):
                    cu  = (C[i][0]*u[0] + C[i][1]*u[1] + C[i][2]*u[2])
                    feq_i = Wt[i]*rho*(1.0 + 3.0*cu + 4.5*cu*cu - 1.5*u2)
                    j = OPP[i]
                    cuj = (C[j][0]*u[0] + C[j][1]*u[1] + C[j][2]*u[2])
                    feq_j = Wt[j]*rho*(1.0 + 3.0*cuj + 4.5*cuj*cuj - 1.5*u2)
                    fp  = 0.5*(f[i,x,y,z] + f[j,x,y,z]);  fep = 0.5*(feq_i + feq_j)
                    fm  = 0.5*(f[i,x,y,z] - f[j,x,y,z]);  fem = 0.5*(feq_i - feq_j)
                    S = 3.0*Wt[i]*ti.cast(C[i][0], fdt)*g
                    fpost[i, x, y, z] = f[i,x,y,z] - om_p*(fp-fep) - om_m*(fm-fem) + S

    @ti.kernel
    def stream_bb():
        # push streaming with periodic wrap
        for x, y, z in ti.ndrange(L, L, L):
            for i in ti.static(range(Q)):
                xn = (x + C[i][0] + L) % L    # +L: '%' on negatives differs across backends
                yn = (y + C[i][1] + L) % L
                zn = (z + C[i][2] + L) % L
                f[i, xn, yn, zn] = fpost[i, x, y, z]
        # full-way bounce-back at solids (after all writes complete)
        ti.loop_config(serialize=False)
        for x, y, z in ti.ndrange(L, L, L):
            if solid[x, y, z] == 1:
                for i in ti.static(range(1, Q, 2)):   # pair heads: (1,2)(3,4)...(17,18)
                    j = OPP[i]
                    tmp = f[i, x, y, z]
                    f[i, x, y, z] = f[j, x, y, z]
                    f[j, x, y, z] = tmp

    qsum = ti.field(ti.f64, shape=())
    umax = ti.field(ti.f64, shape=())

    @ti.kernel
    def measure_q():
        qsum[None] = 0.0
        umax[None] = 0.0
        for x, y, z in ti.ndrange(L, L, L):
            if solid[x, y, z] == 0:
                rho = 0.0
                m = ti.Vector([0.0, 0.0, 0.0])
                for i in ti.static(range(Q)):
                    fi = f[i, x, y, z]
                    rho += fi
                    m += fi * ti.Vector([ti.cast(C[i][0], fdt),
                                         ti.cast(C[i][1], fdt),
                                         ti.cast(C[i][2], fdt)])
                u = m / rho
                qsum[None] += ti.cast(u[0] + 0.5*g, ti.f64)   # f64 accumulation
                ti.atomic_max(umax[None], ti.cast(u.norm(), ti.f64))

    @ti.kernel
    def export_ux():
        for x, y, z in ti.ndrange(L, L, L):
            if solid[x, y, z] == 1:
                ux[x, y, z] = 0.0
            else:
                rho = 0.0; mx = 0.0
                for i in ti.static(range(Q)):
                    rho += f[i, x, y, z]
                    mx += f[i, x, y, z]*C[i][0]
                ux[x, y, z] = mx/rho + 0.5*g

    init_eq()
    q_prev, t0 = None, time.time()
    step = 0
    for step in range(1, max_steps+1):
        collide()
        stream_bb()
        if step % check == 0:
            measure_q()
            q = qsum[None] / (L*L*L)
            um = umax[None]
            if not np.isfinite(q):
                raise FloatingPointError(
                    f"NaN at step {step}: lattice velocity blew up. Reduce g "
                    f"(heterogeneous/channelled packs need g ~ 3e-6; u_max scales "
                    f"as g x channel_width^2 / nu and must stay << 0.1).")
            if um > 0.10:
                print(f"  WARNING step {step}: u_max = {um:.3f} lattice units "
                      f"(Mach ~ {um*3**0.5:.2f}) - reduce g before trusting results")
            if verbose:
                print(f"  step {step:6d}  q = {q:.6e}  u_max = {um:.2e}  ({time.time()-t0:.0f}s)")
            if step >= min_steps and q_prev is not None and abs(q - q_prev) < rtol*abs(q):
                break
            q_prev = q

    if step == max_steps:
        print(f"  WARNING: hit max_steps={max_steps} without meeting rtol={rtol} "
              f"- treat k and the flow field as UNCONVERGED")
    measure_q()
    q = qsum[None] / (L*L*L)
    export_ux()
    phi = 1.0 - solid_np.mean()
    return dict(q=float(q), k=float(nu*q/(g*phi)), phi=float(phi), nu=nu,
                steps=step, seconds=round(time.time()-t0, 1),
                ux=ux.to_numpy())

# Bounce-back swap: pairs are adjacent in this ordering, (1,2)(3,4)...(17,18),
# so iterating the odd pair-heads i = 1,3,...,17 exchanges each pair exactly once.

if __name__ == "__main__":
    import json
    init_lb(arch="cpu")
    # channel validation, identical bookkeeping to mini_lb
    Nz = 41
    solid = np.zeros((Nz, Nz, Nz), dtype=bool)   # cubic for this port
    solid[:, :, 0] = True; solid[:, :, -1] = True
    r = solve(solid, g=1e-6, tau_plus=1.2, max_steps=40000, verbose=False)
    h = float(Nz - 2)
    q_chan = r["q"] * Nz / h
    k_meas = r["nu"] * q_chan / 1e-6
    k_exact = h*h/12.0
    print(f"channel: k = {k_meas:.3f} vs exact {k_exact:.3f}  "
          f"err {100*(k_meas/k_exact-1):+.3f}%  ({r['steps']} steps, {r['seconds']}s)")


In [ ]:
%%writefile puck_pack.py
"""puck_pack.py — synthetic coffee-puck generator + column-flux sigma pipeline.

Step-3 groundwork for the microstructural sigma study:
  make_pack()    Boolean (overlapping-sphere) packing at any target solid fraction,
                 boulder radius from Cameron's microstructure table for a grind
                 setting, with a controllable columnar heterogeneity field standing
                 in for fines-driven clustering (explicit 12 um fines need ~4 um
                 voxels -> FluidX3D-scale domains; flagged, not silently ignored).
  sigma_micro()  Partition an LB velocity field into streamtube columns, measure
                 the per-column flux distribution, fit lognormal sigma, and report
                 the same observables our paper anchors sigma to (CV, top-quartile
                 flow share).

Overlapping spheres are deliberate: the percolation permeability law of
Wadsworth et al. 2026 was validated on exactly this model class (their refs
20, 26), so synthetic packs stay inside the validated universality class.
"""
import numpy as np
from scipy.ndimage import gaussian_filter

def boulder_radius_um(gs):
    """Cameron boulder radius for a grind setting, in microns (standalone-safe)."""
    try:
        import espresso_model as em
        return float(em.grind_microstructure(gs)[2]) * 1e6
    except ImportError:
        import numpy as _np
        return float(_np.interp(gs, [1.0, 1.5, 2.0, 2.5], [273.9, 335.4, 335.4, 410.8]))

def _sphere_stencil(r):
    n = int(np.ceil(r))
    dx, dy, dz = np.meshgrid(*(np.arange(-n, n+1),)*3, indexing="ij")
    m = dx*dx + dy*dy + dz*dz <= r*r
    return dx[m], dy[m], dz[m]

def hetero_field(L, corr_len, seed):
    """Columnar (y,z) heterogeneity field, zero mean, unit std, periodic."""
    rng = np.random.default_rng(seed)
    h = gaussian_filter(rng.standard_normal((L, L)), corr_len, mode="wrap")
    return (h - h.mean()) / h.std()

def make_pack(L=40, voxel_um=30.0, gs=1.3, phis_target=0.55,
              hetero_amp=0.0, hetero_len=8, seed=0, batch=64, verbose=True,
              r_um=None, w_floor=0.25):
    """Periodic boolean pack. Boulder radius a2(gs) from Cameron's table.

    hetero_amp modulates local placement probability with a columnar field:
    amp=0 -> statistically uniform; amp~1-2 -> strongly channelled bed.
    Returns (solid, meta).
    """
    try:
        import espresso_model as em
        phi1, phi2, a2, _, _ = em.grind_microstructure(gs)
    except ImportError:                      # standalone (e.g. Colab): baked-in table
        _GS  = [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5)]
        _A2  = [np.float64(273.9), np.float64(335.4), np.float64(335.4), np.float64(410.8)]
        _PH1 = [np.float64(0.1689), np.float64(0.1343), np.float64(0.12), np.float64(0.078)]
        import numpy as _np
        a2   = _np.interp(gs, _GS, _A2) * 1e-6
        phi1 = _np.interp(gs, _GS, _PH1)
        phi2 = 0.0
    if r_um is not None:                    # explicit grain radius (e.g. Wadsworth <R>)
        a2 = r_um * 1e-6
    r_vox = a2*1e6 / voxel_um
    if 12.0/voxel_um >= 1.0/3.0:
        fines_note = ("fines (a1 = 12 um) are sub-voxel at %.0f um resolution; "
                      "represented implicitly via the heterogeneity field" % voxel_um)
    else:
        fines_note = "resolution admits explicit fines (not implemented in this demo)"

    rng = np.random.default_rng(seed)
    H = hetero_field(L, hetero_len, seed+1)
    # Weight floor: channels are preferential paths THROUGH grains, not empty
    # pipes -- and open slabs also make LB spin-up time scale as width^2.
    w = np.clip(1.0 + hetero_amp*H, w_floor, None)   # acceptance weight per (y,z)
    w /= w.max()

    dx, dy, dz = _sphere_stencil(r_vox)
    solid = np.zeros((L, L, L), dtype=bool)
    n_placed = 0
    done = False
    while not done:
        cx = rng.integers(0, L, batch)
        cy = rng.integers(0, L, batch)
        cz = rng.integers(0, L, batch)
        keep = rng.random(batch) < w[cy, cz]
        for x0, y0, z0 in zip(cx[keep], cy[keep], cz[keep]):
            solid[(x0+dx) % L, (y0+dy) % L, (z0+dz) % L] = True
            n_placed += 1
            if solid.mean() >= phis_target:
                done = True
                break
    meta = dict(L=L, voxel_um=voxel_um, gs=gs, a2_um=a2*1e6, r_vox=float(r_vox),
                phis=float(solid.mean()), phi=float(1-solid.mean()),
                n_spheres=n_placed, hetero_amp=hetero_amp, hetero_len=hetero_len,
                fines_note=fines_note, seed=seed)
    if verbose:
        print(f"pack: L={L} ({L*voxel_um/1000:.2f} mm), boulder r={r_vox:.1f} vox, "
              f"{n_placed} spheres, phis={solid.mean():.3f}, amp={hetero_amp}")
    return solid, meta

def sigma_micro(ux, solid, ncol=4, g=None):
    """Streamtube-column flux statistics from an LB velocity field (flow along x).

    Partitions the (y,z) cross-section into ncol x ncol columns, computes each
    column's superficial flux, normalizes to unit mean, and reports:
      sigma  - std of ln(flux)  (the lognormal spread our ensemble model uses)
      cv     - coefficient of variation of column flux
      top25  - fraction of total flow carried by the fastest quarter of columns
    """
    L = ux.shape[1]
    u = ux.copy(); u[solid] = 0.0
    edges = np.linspace(0, L, ncol+1).astype(int)
    q = np.zeros((ncol, ncol))
    for i in range(ncol):
        for j in range(ncol):
            q[i, j] = u[:, edges[i]:edges[i+1], edges[j]:edges[j+1]].mean()
    k = q.ravel() / q.mean()
    kpos = np.clip(k, 1e-6, None)
    sig = float(np.std(np.log(kpos), ddof=1))
    cv = float(np.std(k, ddof=1))
    ks = np.sort(k)[::-1]
    n25 = max(1, len(k)//4)
    top25 = float(ks[:n25].sum() / ks.sum())
    return dict(sigma=sig, cv=cv, top25=top25, columns=k.round(3).tolist())

if __name__ == "__main__":
    import json, sys
    amp = float(sys.argv[1]) if len(sys.argv) > 1 else 0.0
    tag = sys.argv[2] if len(sys.argv) > 2 else ("A" if amp == 0 else "B")
    solid, meta = make_pack(L=40, voxel_um=30.0, gs=1.3, phis_target=0.55,
                            hetero_amp=amp, hetero_len=8, seed=42)
    np.savez_compressed(f"/tmp/pack_{tag}.npz", solid=solid, meta=json.dumps(meta))
    print("saved /tmp/pack_%s.npz  |  %s" % (tag, meta["fines_note"]))


In [ ]:
# ---------------- configuration ----------------
ARCH, DTYPE = "gpu", "f32"
G_FORCE     = 1e-4   # homogeneous packs / validation (u stays tiny)
G_FORCE_HET = 3e-6
TAU_HET     = 1.8    # nu = 0.433 for heterogeneous runs (TRT magic keeps walls honest)   # heterogeneous/channelled packs: u_max ~ g*d^2/nu must stay << 0.1,
                     # and channel width d grows with domain size and hetero_len
PRESET = "A100"         # "T4" (16 GB) or "A100" (40-80 GB, Colab Pro)
QUICK = True            # False -> full grids
SIZES = {"T4":   dict(L_A_q=192, L_A=256, L_B_q=200, L_B=300, NCOL=5),
         "A100": dict(L_A_q=256, L_A=320, L_B_q=320, L_B=600, NCOL=6)}[PRESET]
OUTDIR = "results"
import os, json, time
import numpy as np
os.makedirs(OUTDIR, exist_ok=True)
import taichi_lb as tlb
from puck_pack import make_pack, sigma_micro
# Optional persistence:
# from google.colab import drive; drive.mount("/content/drive"); OUTDIR = "/content/drive/MyDrive/espresso_lb"

In [ ]:
# ---------------- validation gate (run before trusting anything) ----------------
tlb.init_lb(ARCH, DTYPE)
Nz = 33
solid = np.zeros((Nz, Nz, Nz), dtype=bool); solid[:, :, 0] = True; solid[:, :, -1] = True
r = tlb.solve(solid, g=G_FORCE, tau_plus=1.2, max_steps=16000, check=400,
              rtol=1e-6, min_steps=2000, verbose=False)
h = float(Nz - 2); k_meas = r["nu"]*(r["q"]*Nz/h)/G_FORCE; k_exact = h*h/12
e1 = 100*(k_meas/k_exact - 1)
print(f"channel: err {e1:+.3f}%  (reference +0.05%)  [{r['seconds']}s]")

tlb.init_lb(ARCH, DTYPE)
solid, meta = make_pack(L=40, voxel_um=30.0, gs=1.3, phis_target=0.55,
                        hetero_amp=1.6, hetero_len=8, seed=42, verbose=False)
r = tlb.solve(solid, g=G_FORCE, tau_plus=1.5, max_steps=6000, check=100,
              rtol=1e-5, min_steps=1500, verbose=False)
s = sigma_micro(r["ux"], solid, ncol=2)
e2 = 100*(r["k"]/2.1542 - 1)
print(f"pack B: k {r['k']:.4f} lu (ref 2.1542, err {e2:+.2f}%), sigma {s['sigma']:.3f} (ref 1.051)")
assert abs(e1) < 0.5 and abs(e2) < 1.0, "VALIDATION GATE FAILED - do not run sweeps"
print("GATE PASSED")

In [ ]:
# ---------------- convergence-doubling gate (run once per preset) ----------------
# Large-domain steady state rides on pore-scale physics, not domain-scale diffusion;
# this verifies that empirically at your production size before trusting the sweeps.
from puck_pack import boulder_radius_um
L_chk = SIZES["L_B_q"]
tlb.init_lb(ARCH, DTYPE)
# hetero_len is a grain-clump property (fixed in grain diameters), NOT a bed
# fraction; w_floor keeps channels as preferential paths through grains rather
# than empty pipes -- both make convergence time domain-size independent.
solid, meta = make_pack(L=L_chk, voxel_um=boulder_radius_um(1.3)/10.0, gs=1.3,
                        phis_target=0.60, hetero_amp=1.5, hetero_len=40,
                        w_floor=0.25, seed=7, verbose=False)
r1 = tlb.solve(solid, g=G_FORCE_HET, tau_plus=TAU_HET, max_steps=12000, check=100,
               rtol=1e-5, min_steps=3000, verbose=False)
tlb.init_lb(ARCH, DTYPE)
r2 = tlb.solve(solid, g=G_FORCE_HET, tau_plus=TAU_HET, max_steps=24000, check=100,
               rtol=1e-7, min_steps=2*max(3000, r1["steps"]), verbose=False)
dk = 100*abs(r1["k"]/r2["k"] - 1)
print(f"k at standard stop {r1['k']:.4f} ({r1['steps']} steps) vs doubled {r2['k']:.4f} "
      f"({r2['steps']} steps): {dk:.2f}% drift")
assert dk < 1.0, "convergence criterion too loose at this scale - raise min_steps"
print("CONVERGENCE GATE PASSED")

In [ ]:
# ---------------- Wadsworth et al. 2026, Table 1 (transcribed) ----------------
# (coffee, G, <R> m, phi_p, s_p 1/m, k m2, err m2)
WADS = [
 ("G",1,2.11e-4,0.4792,33588.43,2.39e-11,2.60e-12),("G",3,3.28e-4,0.4979,24439.88,7.38e-11,7.99e-12),
 ("G",4,3.82e-4,0.4643,17105.00,1.55e-10,9.36e-11),("G",5,4.42e-4,0.3707,21916.45,5.84e-11,2.10e-11),
 ("G",6,4.93e-4,0.6733,28288.25,1.25e-10,1.41e-11),("G",7,5.43e-4,0.5312,29619.70,4.87e-11,1.60e-12),
 ("G",8,6.08e-4,0.5380,33231.10,4.63e-11,1.01e-12),("G",9,6.53e-4,0.5166,32187.77,4.94e-11,3.43e-12),
 ("G",10,7.31e-4,0.5558,36498.73,3.91e-11,4.22e-12),("G",11,8.18e-4,0.5147,46688.25,1.58e-11,1.05e-13),
 ("T",1,1.92e-4,0.4558,22265.34,6.01e-11,9.05e-12),("T",2,2.41e-4,0.4945,28789.66,5.78e-11,3.69e-12),
 ("T",3,3.02e-4,0.4908,22528.85,9.93e-11,5.46e-12),("T",4,3.57e-4,0.5186,23026.25,1.04e-10,2.09e-11),
 ("T",5,4.27e-4,0.4903,18580.90,1.51e-10,1.27e-11),("T",6,4.78e-4,0.5021,19616.52,1.91e-10,7.96e-11),
 ("T",7,5.32e-4,0.5507,23500.01,9.06e-11,1.44e-12),("T",8,5.81e-4,0.5169,31510.84,3.49e-11,1.04e-12),
 ("T",9,6.48e-4,0.5126,35301.57,3.40e-11,3.44e-12),("T",10,7.07e-4,0.5113,38074.56,2.54e-11,1.55e-12),
 ("T",11,7.65e-4,0.5625,39815.76,3.09e-11,2.09e-12)]

# ---------------- Sweep A: synthetic Table-1 replication ----------------
R_VOX = 12                      # grain radius in voxels (validated regime)
L_A   = SIZES["L_A_q"] if QUICK else SIZES["L_A"]
subset = [0, 4, 8, 10, 14, 19] if QUICK else range(len(WADS))
seeds  = [0] if QUICK else [0, 1]
rows = []
for idx in subset:
    cof, G, R, phip, sp, k_meas, k_err = WADS[idx]
    voxel_um = R*1e6 / R_VOX
    for sd in seeds:
        tlb.init_lb(ARCH, DTYPE)                       # frees prior fields
        solid, meta = make_pack(L=L_A, voxel_um=voxel_um, phis_target=1-phip,
                                r_um=R*1e6, hetero_amp=0.0, seed=sd, verbose=False)
        t0 = time.time()
        r = tlb.solve(solid, g=G_FORCE, tau_plus=1.5, max_steps=8000, check=100,
                      rtol=1e-5, min_steps=1500, verbose=False)
        k_sim = r["k"] * (voxel_um*1e-6)**2
        rows.append(dict(coffee=cof, G=G, seed=sd, phip=phip, R_um=R*1e6,
                         k_meas=k_meas, k_sim=k_sim, ratio=k_sim/k_meas,
                         steps=r["steps"], secs=round(time.time()-t0,1)))
        print(f"{cof}{G:>3} seed{sd}: k_sim {k_sim:.2e} vs meas {k_meas:.2e} "
              f"(x{k_sim/k_meas:.2f})  [{rows[-1]['secs']}s]")
json.dump(rows, open(f"{OUTDIR}/sweepA_table1.json", "w"), indent=1)
import numpy as _np
ratios = _np.array([r["ratio"] for r in rows])
print(f"\ngeometric-mean k_sim/k_meas = {_np.exp(_np.mean(_np.log(ratios))):.2f} "
      f"(x/÷{_np.exp(_np.std(_np.log(ratios))):.2f})  -- offset from grain angularity expected")

In [ ]:
# ---------------- Sweep B: sigma_micro production study ----------------
GRINDS = [1.3] if QUICK else [1.1, 1.3, 1.5]
# QUICK pass showed sigma* = 0.22-0.57 maps to amp ~ 0.10-0.25: sample THAT decade,
# with 0.5 as a bridge anchor to the QUICK results.
AMPS   = [0.0, 1.0, 2.0] if QUICK else [0.0, 0.05, 0.10, 0.15, 0.25, 0.5]
SEEDS  = [0] if QUICK else [0, 1]
SEEDS_FLOOR = [0] if QUICK else [0, 1, 2]     # extra seeds at amp=0 for the noise floor
L_B    = SIZES["L_B_q"] if QUICK else SIZES["L_B"]
R_VOX_B = 10
NCOL   = SIZES["NCOL"]
SIGMA_STAR = {1.1: 0.569, 1.3: 0.359, 1.5: 0.224}     # calibrated (extraction-based)

out = []
for gs in GRINDS:
    for amp in AMPS:
        for sd in (SEEDS_FLOOR if amp == 0.0 else SEEDS):
            tlb.init_lb(ARCH, DTYPE)
            from puck_pack import boulder_radius_um
            voxel_um = boulder_radius_um(gs) / R_VOX_B   # boulders = R_VOX_B voxels
            solid, meta = make_pack(L=L_B, voxel_um=voxel_um, gs=gs,
                                    phis_target=0.60, hetero_amp=amp,
                                    hetero_len=4*R_VOX_B, w_floor=0.25,
                                    seed=100+sd, verbose=False)
            r = tlb.solve(solid, g=G_FORCE_HET, tau_plus=TAU_HET, max_steps=12000,
                          check=100, rtol=1e-5, min_steps=3000, verbose=False)
            s = sigma_micro(r["ux"], solid, ncol=NCOL)
            colw = L_B/NCOL/(2*meta["r_vox"])
            out.append(dict(gs=gs, amp=amp, seed=sd, k=r["k"], steps=r["steps"],
                            colw_diam=round(colw,2),
                            **{k2: v for k2, v in s.items() if k2 != "columns"}))
            print(f"G{gs} amp{amp} seed{sd}: sigma {s['sigma']:.3f}  CV {s['cv']:.3f} "
                  f" top25 {s['top25']:.0%}  k {r['k']:.2f} lu  (cols {colw:.1f} grain diam)")
json.dump(out, open(f"{OUTDIR}/sweepB_sigma.json", "w"), indent=1)
print("\ncalibrated targets:", SIGMA_STAR)

In [ ]:
# ---------------- plots ----------------
import matplotlib.pyplot as plt
import numpy as np, json
rows = json.load(open(f"{OUTDIR}/sweepA_table1.json"))
out  = json.load(open(f"{OUTDIR}/sweepB_sigma.json"))

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
km = [r["k_meas"] for r in rows]; ks = [r["k_sim"] for r in rows]
axs[0].loglog(km, ks, "o", mfc="none")
lim = [min(km+ks)*0.5, max(km+ks)*2]
axs[0].loglog(lim, lim, "k--", lw=1)
axs[0].set_xlabel("measured k (Wadsworth Table 1), m$^2$")
axs[0].set_ylabel("simulated k (synthetic spheres), m$^2$")
axs[0].set_title("Sweep A: Table-1 replication")

for gs in sorted(set(o["gs"] for o in out)):
    pts = [(o["amp"], o["sigma"]) for o in out if o["gs"] == gs]
    a = sorted(set(p[0] for p in pts))
    m = [np.mean([p[1] for p in pts if p[0] == ai]) for ai in a]
    axs[1].plot(a, m, "o-", label=f"G {gs}")
    axs[1].axhline({1.1:0.569, 1.3:0.359, 1.5:0.224}[gs], ls=":", lw=1)
axs[1].set_xlabel("heterogeneity amplitude"); axs[1].set_ylabel(r"$\sigma_{micro}$")
axs[1].set_title("Sweep B: column-flux spread vs amplitude\n(dotted: calibrated $\sigma^*$)")
axs[1].legend()
fig.tight_layout(); fig.savefig(f"{OUTDIR}/sweeps.png", dpi=150)
plt.show()
print("saved", OUTDIR)

In [ ]:
# ---------------- future: Wadsworth XCT volumes (when the data arrives) ----------------
def load_xct(path_or_glob, threshold=None, crop=None):
    """TIFF stack -> boolean solid array. threshold=None assumes already binary."""
    import tifffile, glob
    files = sorted(glob.glob(path_or_glob)) if "*" in path_or_glob else [path_or_glob]
    vol = tifffile.imread(files if len(files) > 1 else files[0])
    solid = (vol > threshold) if threshold is not None else vol.astype(bool)
    if crop:
        c = crop
        solid = solid[c[0]:c[0]+c[3], c[1]:c[1]+c[3], c[2]:c[2]+c[3]]
    return solid

# usage once volumes arrive:
# solid = load_xct("/content/drive/MyDrive/wadsworth/G5_*.tif", threshold=0)
# L = min(solid.shape); solid = solid[:L, :L, :L]      # solver expects cubic
# tlb.init_lb(ARCH, DTYPE)
# r = tlb.solve(solid, g=G_FORCE, tau_plus=1.5, rtol=1e-5, min_steps=1500)
# print(r["k"] * (voxel_um*1e-6)**2, "m2   vs their Table 1")
print("XCT loader ready")

**Memory (f32, two population fields):** L=300 → 8 GB · L=400 → 10 GB ·
L=512 → 20 GB · L=600 → 33 GB · L=690 → 50 GB. f64 doubles these (A100: spot-check
f32 against f64 at L ≤ 500 by setting `DTYPE="f64"` for one run).

**Full-grid runtime (A100, 600³):** Sweep A full (42 runs) ~15–25 min. Sweep B full
(3 grinds × 6 amps × 2 seeds + extra floor seeds ≈ 39 runs) ~4–5 h at ~7 min/run —
an afternoon job; mount Drive first so results survive a disconnect.

**Heterogeneous-pack recipe (v3):** correlation length fixed at 2 grain diameters
(a clump property, not a bed fraction) and placement weight floored at 0.25 —
channels are preferential paths through grains, not empty pipes. Verified: drift
0.01% at 2k/4k steps vs 10.35% at 8k/16k for the old open-pipe packs.

**Driving force rule:** peak lattice speed scales as g × (channel width)² / ν and
must stay well below 0.1. Homogeneous packs: `G_FORCE = 1e-4`. Channelled packs
(any `hetero_amp > 0`): `G_FORCE_HET = 3e-6`. The solver now raises a clear error
on NaN and warns above u_max = 0.1.

**Runtime guide:** T4 — gate ~1 min, QUICK sweeps ~25–35 min, full grids a few hours.
A100 — QUICK sweeps ~10 min; full Sweep B at 600³ roughly 5–10 min/run × 45 runs.
Each run re-initializes Taichi (frees GPU fields) at ~5–10 s JIT cost.

**Sending results back:** the three JSONs + `sweeps.png` in `results/` are all the
analysis needs — download and drop them into a Claude session.